# Étape 5 — Évaluation sur le test (version Jupyter)

Ce notebook est l'équivalent de `05_evaluation.py`. C'est la première et
unique utilisation du jeu de test : les étapes 1 à 4 ne l'ont jamais lu pour
choisir un modèle ou régler un paramètre.

Règles conservées :

- graine `RANDOM_STATE = 42` implicitement respectée par les objets chargés ;
- modèle final chargé depuis `models/final_model.joblib` ;
- métriques, rapport par famille, matrice de confusion et erreurs écrits dans
  `reports/` ;
- après ce notebook, le test ne sert plus à choisir quoi que ce soit.

In [ ]:
import os
import sys
from pathlib import Path

# Jupyter ouvre le notebook depuis notebooks/ : on remonte jusqu'à config.py
dossier = Path.cwd()
while dossier != dossier.parent and not (dossier / "config.py").exists():
    dossier = dossier.parent
os.chdir(dossier)
if str(dossier) not in sys.path:
    sys.path.insert(0, str(dossier))
print("Dossier du projet :", dossier)

In [ ]:
%matplotlib inline

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, auc, balanced_accuracy_score,
                             classification_report, confusion_matrix, f1_score,
                             precision_recall_curve, precision_score, recall_score)

from config import (PROCESSED_DIR, FIGURES_DIR, TABLES_DIR,
                    MODELS_DIR, FAMILY_ORDER)


def pr_auc_macro(y_true, y_score, classes):
    # Aire sous la courbe précision-rappel de chaque classe, puis moyenne
    y_binaire = (np.asarray(y_true)[:, None] == np.asarray(classes)).astype(int)
    aires = []
    for i in range(len(classes)):
        precision, rappel, _ = precision_recall_curve(y_binaire[:, i], y_score[:, i])
        aires.append(auc(rappel, precision))
    return float(np.mean(aires))

## 1. Chargement du test et du modèle final

On relit les 15 variables de l'étape 3, les 27 197 flux du test, le pipeline
sans SMOTE et l'encodeur des familles. Les prédictions sont calculées en une
seule passe.

In [ ]:
with open(PROCESSED_DIR / "selected_features.txt", encoding="utf-8") as fichier:
    colonnes = [ligne.strip() for ligne in fichier if ligne.strip()]

test = pd.read_csv(PROCESSED_DIR / "test.csv", usecols=colonnes + ["Family"])
X_test = test[colonnes]
y_vrai = test["Family"]

modele = joblib.load(MODELS_DIR / "final_model.joblib")
encodeur = joblib.load(MODELS_DIR / "label_encoder.joblib")

y_code = encodeur.transform(y_vrai)
y_pred_code = modele.predict(X_test)
probabilites = modele.predict_proba(X_test)

y_pred = encodeur.inverse_transform(y_pred_code)
classes = list(range(len(encodeur.classes_)))
print("Test :", len(X_test), "flux,", len(colonnes), "variables,",
      len(classes), "familles")

## 2. Métriques globales

L'accuracy brute est portée par la majorité `BENIGN`. On regarde surtout le F1
macro, le rappel macro, la précision macro et le PR-AUC macro.

In [ ]:
metriques = {
    "accuracy": round(accuracy_score(y_code, y_pred_code), 4),
    "balanced_accuracy": round(balanced_accuracy_score(y_code, y_pred_code), 4),
    "f1_macro": round(f1_score(y_code, y_pred_code, average="macro", zero_division=0), 4),
    "f1_pondere": round(f1_score(y_code, y_pred_code, average="weighted", zero_division=0), 4),
    "precision_macro": round(precision_score(y_code, y_pred_code, average="macro", zero_division=0), 4),
    "rappel_macro": round(recall_score(y_code, y_pred_code, average="macro", zero_division=0), 4),
    "pr_auc_macro": round(pr_auc_macro(y_code, probabilites, classes), 4),
}
table_metriques = pd.DataFrame([metriques])
table_metriques.to_csv(TABLES_DIR / "metriques_test.csv", index=False)
table_metriques

## 3. Résultats par famille

Le rapport `classification_report` est converti en tableau lisible. Les
dernières lignes sont les moyennes macro et pondérée.

In [ ]:
rapport = classification_report(y_code, y_pred_code, labels=classes,
                                target_names=encodeur.classes_,
                                output_dict=True, zero_division=0)
lignes = []
for famille, valeurs in rapport.items():
    if isinstance(valeurs, dict):
        lignes.append({"famille": famille,
                       "precision": round(valeurs["precision"], 4),
                       "rappel": round(valeurs["recall"], 4),
                       "f1": round(valeurs["f1-score"], 4),
                       "support": int(valeurs["support"])})
par_famille = pd.DataFrame(lignes)
par_famille.to_csv(TABLES_DIR / "rapport_classification.csv", index=False)
par_famille

## 4. Comparaison avec la validation croisée

On reprend le meilleur résultat de `comparaison_apres_reglage.csv` (étape 4).
Le test est légèrement au-dessus de la CV : le modèle final a vu tout le train,
contre 4/5 à chaque fold.

In [ ]:
cv = pd.read_csv(TABLES_DIR / "comparaison_apres_reglage.csv")
meilleur_cv = cv.sort_values("f1_macro_moy", ascending=False).iloc[0]
comparaison = pd.DataFrame([
    {"jeu": "CV 5 folds (train)", "f1_macro": meilleur_cv["f1_macro_moy"],
     "pr_auc_macro": meilleur_cv["pr_auc_macro_moy"]},
    {"jeu": "Test intact", "f1_macro": metriques["f1_macro"],
     "pr_auc_macro": metriques["pr_auc_macro"]},
])
comparaison.to_csv(TABLES_DIR / "comparaison_test_cv.csv", index=False)
comparaison

## 5. Matrice de confusion

Les familles sont remises dans l'ordre du projet (`FAMILY_ORDER`) pour que la
lecture soit cohérente avec les autres étapes.

In [ ]:
matrice = confusion_matrix(y_code, y_pred_code, labels=classes)
ordre = [famille for famille in FAMILY_ORDER if famille in encodeur.classes_]
_indices = [list(encodeur.classes_).index(famille) for famille in ordre]
matrice_ordonee = matrice[np.ix_(_indices, _indices)]

fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(matrice_ordonee, cmap="Blues")
ax.set_xticks(range(len(ordre)), ordre, rotation=45, ha="right")
ax.set_yticks(range(len(ordre)), ordre)
ax.set_xlabel("Famille prédite")
ax.set_ylabel("Famille réelle")
ax.set_title("Matrice de confusion sur le test")
for i in range(len(ordre)):
    for j in range(len(ordre)):
        ax.text(j, i, str(matrice_ordonee[i, j]), ha="center", va="center",
                color="white" if matrice_ordonee[i, j] > matrice_ordonee.max() / 2 else "black",
                fontsize=8)
fig.colorbar(image, ax=ax, shrink=0.8)
plt.show()
fig.savefig(FIGURES_DIR / "matrice_confusion_test.png", dpi=120, bbox_inches="tight")
plt.close(fig)

## 6. F1 par famille

Une barre par famille permet de voir tout de suite les points faibles, en
particulier les fausses alertes sur du trafic normal.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(par_famille["famille"], par_famille["f1"], color="steelblue")
ax.set_ylim(0, 1)
ax.set_ylabel("F1")
ax.set_title("F1 par famille sur le test")
ax.tick_params(axis="x", rotation=30)
plt.show()
fig.savefig(FIGURES_DIR / "f1_par_famille_test.png", dpi=120, bbox_inches="tight")
plt.close(fig)

## 7. Erreurs sauvegardées pour l'étape 6

On garde chaque flux mal classé avec sa famille réelle, sa famille prédite et
sa confiance. L'étape 6 analysera ces erreurs sans toucher au modèle.

In [ ]:
masque_erreur = y_pred_code != y_code
erreurs = test.loc[masque_erreur, colonnes].copy()
erreurs.insert(0, "confiance", probabilites[masque_erreur].max(axis=1).round(4))
erreurs.insert(0, "famille_predite", y_pred[masque_erreur])
erreurs.insert(0, "famille_reelle", y_vrai[masque_erreur].values)
erreurs.to_csv(TABLES_DIR / "erreurs_test.csv", index=False)
print("\nErreurs sauvegardées :", len(erreurs))
erreurs.head(10)

## Résumé de l'étape 5 (version Jupyter)

Résultats identiques à ceux du script `05_evaluation.py` :

- 27 197 flux de test, 15 variables, 7 familles ;
- F1 macro **0,8926**, PR-AUC macro 0,8897, accuracy 0,9966,
  accuracy équilibrée 0,9907 ;
- précision macro 0,8481, rappel macro 0,9907 ;
- 93 erreurs sauvegardées dans `erreurs_test.csv` ;
- `WebAttack` a une précision de 0,3729 et `Bot` de 0,5806, malgré un rappel
  parfait ;
- le test est légèrement au-dessus de la CV (0,8714) car le modèle final a vu
  tout le train.

In [ ]:
print("\nFin de l'étape 5 : le test n'est plus réutilisé pour choisir un modèle.")